In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
import pandas as pd

data = fetch_20newsgroups(subset='all', categories=['sci.med', 'sci.space'], remove=('headers', 'footers', 'quotes'))
df = pd.DataFrame({'text': data.data, 'label': data.target})
print(df.shape)
print(df['label'].value_counts())

(1977, 2)
label
0    990
1    987
Name: count, dtype: int64


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

# Vectorize
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=['sci.med', 'sci.space']))

              precision    recall  f1-score   support

     sci.med       0.91      0.92      0.91       193
   sci.space       0.92      0.92      0.92       203

    accuracy                           0.92       396
   macro avg       0.92      0.92      0.92       396
weighted avg       0.92      0.92      0.92       396



In [3]:
# What words most strongly predict each class?
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

import numpy as np
top_space = np.argsort(coefficients)[-10:]
top_med = np.argsort(coefficients)[:10]

print("Top words → sci.space:")
print([feature_names[i] for i in top_space])

print("\nTop words → sci.med:")
print([feature_names[i] for i in top_med])

Top words → sci.space:
['spacecraft', 'sky', 'launch', 'shuttle', 'earth', 'nasa', 'orbit', 'moon', 'the', 'space']

Top words → sci.med:
['my', 'medical', 'msg', 'she', 'your', 'is', 'doctor', 'disease', 'me', 'cancer']


TF-IDF
Converts text into numeric vectors using word frequency, penalizing words that appear across all documents. No neural network — pure statistics. Each document becomes a fixed-size vector of word importance scores.
Why fit only on train
fit learns the vocabulary. Fitting on test data leaks information — the vectorizer would adapt to test words, making evaluation unreliable. Fit on train, transform everything else.
Stopwords problem
Top predictors for sci.med included my, she, your, is, me — meaningless filler words. The model partially learned conversational tone instead of medical content. Remove stopwords before vectorizing to force the model to learn real signal.
Classical NLP ceiling vs embeddings
TF-IDF has no understanding of meaning — cancer and tumor are unrelated to it. nasa and space agency are completely different vectors. Distilbert breaks this ceiling because semantically similar words cluster close together in vector space, learned from billions of examples.